<img src="images/logo.png" width="15%" height="15%">

# **LangChain Essentials**

### **3. Tools**

LangChain simplifies functions calling by automating your tools registration via a Python decorator.

There is no need to write a lengthy dictionary for each function any more (but still possible though)::

```python

def adder(x: int, y: int) -> int:
    return x+y

toolbox = [
    {
        "type": "function",
        "function": {
            "name":"adder",
            "description":"Returns the sum of 2 integers",
            "parameters":{
                "type": "object",
                "properties": {
                    "x": {
                        "type": "number",
                        "description": "The first integer to sum-up",
                    },
                    "y": {
                        "type": "number",
                        "description": "The second integer to sum-up",
                    }
                },
                "required": ["x", "y"]
            }
        }
    }
]
```

Instead, the `@tool` decorator extracts your docstring and type hints to document your function.

In [19]:
from langchain.tools import tool

@tool
def adder(x: int, y: int) -> int:
    """
    Returns the sum of two integers
    args:
      x: first integer to sum up
      y: second integer to sum up
    """
    return x+y

# List of all tools
toolbox = [adder]

# The tools registry simplifies the tool selection when processing tool calls (=> can select a tool by name)
tools_registry = {tool.name: tool for tool in toolbox}
tools_registry

{'adder': StructuredTool(name='adder', description='Returns the sum of two integers\nargs:\n  x: first integer to sum up\n  y: second integer to sum up', args_schema=<class 'langchain_core.utils.pydantic.adder'>, func=<function adder at 0x11062fb60>)}

In [20]:
print(type(adder), end="\n\n")
print(adder.description)

<class 'langchain_core.tools.structured.StructuredTool'>

Returns the sum of two integers
args:
  x: first integer to sum up
  y: second integer to sum up


In [21]:
# Once this is a tool, you can't call it as a Python function: adder(x=5, y=8) would not work
# Use the invoke() method like any LangChain runnables and pass a dict with arguments
adder.invoke({"x": 5, "y": 8})

13

In [22]:
from langchain_mistralai import ChatMistralAI
from dotenv import load_dotenv

load_dotenv()
llm = ChatMistralAI(model_name="mistral-small-latest")

# We create a second LLM client which has access to tools
llm_with_tools = llm.bind_tools(toolbox)

In [23]:
from langchain_core.messages import SystemMessage, HumanMessage

memory = [
    SystemMessage(content="You are a helpful assistant using tools to respond to questions"),
    HumanMessage(content="How much is 743+999?")
]

response = llm_with_tools.invoke(memory)

In [24]:
response.content

''

In [25]:
# Not response.choices[0].message.tool_calls any more!
response.tool_calls

[{'name': 'adder',
  'args': {'x': 743, 'y': 999},
  'id': 'ggwG0qyvo',
  'type': 'tool_call'}]

In [26]:
from langchain_core.messages import ToolMessage

memory.append(response)

for tool_call in response.tool_calls:
    name = tool_call["name"]
    args = tool_call["args"] # Already a dict! No need for json.loads()
    id = tool_call["id"]
    output = tools_registry[name].invoke(args)
    memory.append(
        ToolMessage(
            content=str(output),
            tool_call_id=id,
            name=name # Optional
        )
    )

In [27]:
# You could also re-invoke the LLM which doesn't have access to tools if you wanted to make sure that no further tool calls would be triggered
response = llm_with_tools.invoke(memory)
memory.append(response)

In [28]:
for msg in memory:
    msg.pretty_print()

================================ System Message ================================

You are a helpful assistant using tools to respond to questions
================================ Human Message =================================

How much is 743+999?
================================== Ai Message ==================================
Tool Calls:
  adder (ggwG0qyvo)
 Call ID: ggwG0qyvo
  Args:
    x: 743
    y: 999
================================= Tool Message =================================
Name: adder

1742
================================== Ai Message ==================================

The sum of **743 + 999** is **1,742**.
